# 04. Face Recognition Pipeline (CVLface / MSU CVLab)

## 1. Overview
- **Task**: Facial Detection, Alignment, and Recognition
- **Tool**: CVLface (Michigan State University Computer Vision Lab)
- **License**: MIT (Permissive commercial license)
- **Key Models**:
  1. Face Landmark / Detection: MediaPipe / YuNet
  2. Feature Extraction: CVLface ONNX (512-d L2-normalized embedding)

### Why CVLface?
- Provides state-of-the-art open-source face representations trained on large-scale datasets.
- Exportable to fixed-shape ONNX, running efficiently via AMD MIGraphX or CPU.
- Multi-shot consensus voting mitigates CCTV motion blur and extreme head poses.

In [ ]:
# Dependencies
import os
import cv2
import numpy as np
import onnxruntime as ort
import matplotlib.pyplot as plt

## 2. Face Alignment & Embedding Extraction

In [ ]:
class FaceEmbedder:
    """CVLface ONNX feature extractor producing 512-d L2-normalized embeddings."""

    INPUT_SIZE = (112, 112) # Standard face crop size (W, H)

    def __init__(self, model_path: str = None):
        self.session = None
        if model_path and os.path.exists(model_path):
            providers = ['MIGraphXExecutionProvider', 'CPUExecutionProvider']
            self.session = ort.InferenceSession(model_path, providers=providers)
            self.input_name = self.session.get_inputs()[0].name
        else:
            print('CVLface model not found. Running in simulation mode.')

    def preprocess(self, bgr_face_crop: np.ndarray) -> np.ndarray:
        # Normalize to standard 112x112 RGB float32 in [-1, 1]
        rgb = cv2.cvtColor(bgr_face_crop, cv2.COLOR_BGR2RGB)
        resized = cv2.resize(rgb, self.INPUT_SIZE, interpolation=cv2.INTER_LINEAR)
        blob = (resized.astype(np.float32) - 127.5) / 128.0
        blob = np.transpose(blob, (2, 0, 1))
        return np.expand_dims(blob, axis=0)

    def embed(self, bgr_face_crop: np.ndarray) -> np.ndarray:
        if self.session is None or bgr_face_crop.size == 0:
            vec = np.random.randn(512).astype(np.float32)
            return vec / (np.linalg.norm(vec) + 1e-6)

        blob = self.preprocess(bgr_face_crop)
        feat = self.session.run(None, {self.input_name: blob})[0][0]
        return feat / (np.linalg.norm(feat) + 1e-6)

## 3. Gallery Enroller & Matcher with Multi-Shot Consensus

In [ ]:
class FaceMatcher:
    """Enrolls known identities and performs cosine similarity search."""

    def __init__(self, match_thresh: float = 0.55):
        self.match_thresh = match_thresh
        self.gallery = {} # person_name -> list of 512-d embeddings

    def enroll(self, name: str, embeddings: list):
        self.gallery[name] = embeddings

    def match(self, query_embedding: np.ndarray):
        best_name = None
        best_score = -1.0

        for name, emb_list in self.gallery.items():
            # Compute cosine similarities
            scores = [float(np.dot(emb, query_embedding)) for emb in emb_list]
            max_score = max(scores)
            if max_score > best_score:
                best_score = max_score
                best_name = name

        if best_score >= self.match_thresh:
            return {'person_id': best_name, 'score': best_score, 'is_known': True}
        else:
            return {'person_id': 'Unknown', 'score': best_score, 'is_known': False}

## 4. Enrollment & Matching Demo

In [ ]:
embedder = FaceEmbedder()
matcher = FaceMatcher(match_thresh=0.60)

# Enrolling Known Staff Member "Somchai"
base_vector = np.random.randn(512).astype(np.float32)
base_vector /= np.linalg.norm(base_vector)
matcher.enroll('Somchai', [base_vector])

# Query 1: High-similarity shot of Somchai with small pose variance
query_somchai = base_vector + np.random.normal(0, 0.05, 512).astype(np.float32)
query_somchai /= np.linalg.norm(query_somchai)
res1 = matcher.match(query_somchai)
print(f"Query 1 Result: Matched '{res1['person_id']}' with score {res1['score']:.3f} (Known: {res1['is_known']})")

# Query 2: Unenrolled stranger
query_stranger = np.random.randn(512).astype(np.float32)
query_stranger /= np.linalg.norm(query_stranger)
res2 = matcher.match(query_stranger)
print(f"Query 2 Result: Matched '{res2['person_id']}' with score {res2['score']:.3f} (Known: {res2['is_known']})")